In [0]:
%sql
USE workspace.metropulse;

In [0]:
from pyspark.sql import functions as F

def missing_report(table):
    df = spark.table(table)
    cols = [c for c in df.columns if not c.startswith("_")]
    checks = [
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in cols
    ]
    print(table)
    display(df.select(checks))

for t in ["bronze_routes", "bronze_trips", "bronze_stops", "bronze_stop_times"]:
    missing_report(t)

bronze_routes


route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,0,0,0,120,0,6,0,0


bronze_trips


route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed,ots_trip_short_name
0,0,0,0,1721,0,0,0,0,0,0


bronze_stops


stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,0,0,20,0,0,8797,8797,8797,8797,8797,0


bronze_stop_times


trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
0,0,0,0,0,1415650,0,0,21387,0


In [0]:
%sql
SELECT 'routes'     AS tbl, COUNT(*) - COUNT(DISTINCT route_id)              AS duplicates FROM bronze_routes
UNION ALL
SELECT 'trips',             COUNT(*) - COUNT(DISTINCT trip_id)                             FROM bronze_trips
UNION ALL
SELECT 'stops',             COUNT(*) - COUNT(DISTINCT stop_id)                             FROM bronze_stops
UNION ALL
SELECT 'stop_times',        COUNT(*) - COUNT(DISTINCT trip_id, stop_sequence)              FROM bronze_stop_times;

tbl,duplicates
routes,0
trips,0
stops,0
stop_times,0


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM bronze_stop_times st LEFT ANTI JOIN bronze_trips  t ON st.trip_id  = t.trip_id)  AS stop_times_missing_trip,
  (SELECT COUNT(*) FROM bronze_stop_times st LEFT ANTI JOIN bronze_stops  s ON st.stop_id  = s.stop_id)  AS stop_times_missing_stop,
  (SELECT COUNT(*) FROM bronze_trips      t  LEFT ANTI JOIN bronze_routes r ON t.route_id  = r.route_id) AS trips_missing_route;

stop_times_missing_trip,stop_times_missing_stop,trips_missing_route
0,0,0


In [0]:
%sql
SELECT t.service_id,
       c.monday, c.tuesday, c.wednesday, c.thursday, c.friday, c.saturday, c.sunday,
       COUNT(*) AS trips
FROM bronze_trips t
LEFT JOIN bronze_calendar c ON t.service_id = c.service_id
GROUP BY ALL
ORDER BY trips DESC;

service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,trips
6,1,1,1,1,1,0,0,9160
4,0,0,0,0,0,1,0,6359
5,0,0,0,0,0,0,1,6359


In [0]:
%sql
SELECT
  COUNT(*)                                                                AS total_rows,
  SUM(CASE WHEN CAST(split(trim(arrival_time), ':')[0] AS INT) >= 24 THEN 1 ELSE 0 END) AS after_midnight_rows,
  MIN(CAST(split(trim(arrival_time), ':')[0] AS INT))                     AS earliest_hour,
  MAX(CAST(split(trim(arrival_time), ':')[0] AS INT))                     AS latest_hour
FROM bronze_stop_times;

total_rows,after_midnight_rows,earliest_hour,latest_hour
1415650,32653,3,26


In [0]:
%sql
SELECT
  COUNT(*) AS total_stops,
  SUM(CASE WHEN CAST(stop_lat AS DOUBLE) NOT BETWEEN 29.0 AND 30.6
             OR CAST(stop_lon AS DOUBLE) NOT BETWEEN -96.2 AND -94.6 THEN 1 ELSE 0 END) AS outside_houston_area,
  MIN(CAST(stop_lat AS DOUBLE)) AS min_lat, MAX(CAST(stop_lat AS DOUBLE)) AS max_lat,
  MIN(CAST(stop_lon AS DOUBLE)) AS min_lon, MAX(CAST(stop_lon AS DOUBLE)) AS max_lon
FROM bronze_stops;

total_stops,outside_houston_area,min_lat,max_lat,min_lon,max_lon
8797,0,29.528849,30.054341,-95.776249,-94.97971


#### Exploration findings
- Missing values: 0 in all key columns. Fully empty columns (route_desc, zone_id, stop_url, stop_headsign, parent_station) are dropped in Silver.
- Duplicate keys: 0 in routes, trips, stops, stop_times
- Broken references: 0 (every stop_time has a valid trip and stop; every trip has a valid route)
- Services: weekday = service 6 (9,160 trips); Saturday = 4 (6,359); Sunday = 5 (6,359). Services 413, 606, 806 have no trips.
- 58% of trips are weekend trips, so analysis must filter to one service day
- After-midnight rows: 32,653 (hours up to 26), so times are stored as seconds after midnight
- Stops outside Houston area: 0
- parent_station is empty for all stops, so analysis is per stop (no hub grouping)